# Assignment 2 : Around the World of Plotting
### EES 3350/5350 · Python in Earth Science · Monday, October 26, 2026 
Due: Sunday, November 8, at 11:59pm, submitted by pushing to your Assignment 2 repo

> **A note before you start:** This is long, and I know it is. It's an assignment instead of a lab because there's a lot here: eight stops,e ahc with something new. You ahve **two weeks**. A good pace is Stops 1-4 this first week (by Sunday, Nov 1) and Stops 5-8 the second. Commit and push as you go. Start in lab on Monday, and come to office hours if you get stuck. The point isn't to get every figure perfect on the first try. It's to see a lot of different ways to plot data you already know.

This is a lab tour. Each **stop** goes back to data from an earlier lab and shows it in a new way. At every stop you'll meet **one new plotting tool**, with a short example, and then you'll use it.

This assignment is also a **review** of everything since Assignment 1. If a skill in the right-hand column feels shakey, that's the lecture to go back to.
|**stop**|**data from**|**the new tool**|**what it reviews**|
|:--|:--|:--|:--|
|1| Lab 01, Earth's energy balance| warmup: a curve with reference lines|functions with docstrings; `np.linspace()`; math on whole arrays (Lecture 14)|
|2| Lab 02, the Calwood Fire gullies| **grouped bars**, and a **log-log** threshold plot| lissts -> arrays; masks and `~` (Lecture 15); `for` + `zip()`|
|3| Lab 03, factor of safety| **shading a region** with `fill_between`|functinos on arrays; masks; `.any()`; (G1) 2D tables with `np.newaxis` (Lecture 16)|
|4| Lab 04, Nashville precipitation normals| a **second y-axis** with `twinx`|**nested loops**, `break`, and a flag variable (Lectures 11-13)|
|5| Lab 06, Nisqually and Northridge| **log time axes**, and a **map colored by a third variable**|`np.loadtxt()`; masks; `np.isnan()`; **2D arrays**, **`[row, col]`**, **and** `axis=`(Lecture 16)|
|6| Lab 08, the ocean buoys| **error bars**| `read_csv` with dates; `.describe()`; `groupby`; `pd.Timedelta`; new columns; `.loc` date slices; `idxmax()`; `to_csv` (Lectures 17-19)|
|7|  an idea from any earlier lab |a **schematic**: a figure from an idea, not data (Lecture 22)| building arrays piece by piece|
|8| two or more of the stops above| a **multipanel figure that tells one story** |everything|

And at every checkpoint: `git add`, `commit`, `push` (Lab 05).

**Every figure you make must pass the checklist form Lecture 21**. That is part of the grade:
1. every axis is labeled, **with units**
2. a title that says what, where, and when
3. a legend whenever there's more than one line or set of points
4. one kind of unit per y-axis (stop 4 is the one careful exception)
5. colors anyone can tell apart, not just red vs. green
6. readable text
7. compare like with like: shared axes when panels are compared, bars that start at zero.

Stops 1-6 each take about 30-45 minutes. Stops 7 and 8 ask you to make your own choices, so give them more time.

## Getting Started

1. Accept the Assignment 2 link.
2. In an ExploreHub terminal, close your repo into your `assignments` folder
```
   cd ~/LasName_EES3350/assignments
   git clone git@github.com:Python-in-Earth-Science/ees-3350-5350-assignment-02-USERNAME.git
   cd ees-3350-5350-assignment-02-USERNAME
   ls
```
You should see this file and four data files:
- `nisqually_2001.csv`
- `northridge_1994.csv`
- `kaneohe_clean.csv`
- `nh10_clean.csv`
3. Rename this file `assignment02_LastName.ipynb` so that to protect it from any pulled changes.

<div class="alert alert-info">

### Question 0: Import the appropriate libraries.

What libraries do you need to import in order to complete this lab? There should be three of them (at least).
    
</div>

In [ ]:
# your answer here

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## Stop 1 : Lab 01, what sets Earth's temperature?

In Lab 01, you found the temperature a planet would have if it were a perfect **blackbody**, absorbing sunlight and radiating the same energy back out:
$$T = \left(\frac{(1-a)\,\Omega}{4\sigma}\right)^{1/4}$$
where $a$ is the **albedo** (the fraction of sunlight reflected), $\Omega = 1372$ W/m² is the solar constant, and $\sigma = 5.67 \times 10^{-8}$ W/(m²K⁴). With Earth's albedo of about 0.30, you got about **255 K**, much colder than Earth's actual average surface temperature of about **288 K**.

Back then you could only plug in one albedo at a time. With NumPy, you can plug in a hundred at once and **see ** the whole relationship.

Nothing new here!

<div class="alert alert-info">

### Exercise 1 : temperature vs. abledo

1. Write a function `blackbody_temp(abledo)` with a docstring that regutns the temperature in K.
    - Check it: `blackbody_temp(0.30) should be about 255.
2. Make `albedos = np.linspace(0, 0.9, 91)` and pass the **whole array** to your function to get `temps`.
3. Plot `temps` against `albedos` as a line. Add:
    - a dashed **horizontal** line at Earth's actual surface temperature, 288 K
    - a dotted **vertical** line at Earth's albedo, 0.30
    - a label for all three lines, and a legend
4. Pass the checklist: labels with units, a title
    
</div>

In [2]:
# your answer here

<div class="alert alert-warning">

### Question 1
Does the blackbody prediction line ever reach Earth's actual surface temperature? what is the **highest** temperature a blackbody Earth could have, and what albedo goes it? What does that tell you about whether albedo alone can explain Earth's real temperature?
    
</div>

> your answer here

---
## Stop 2 : Lab 02, the Calwood Fire gullies

In Lab 02, you stored field measurements from four gullies burned in the 2020 Calwood Fire in lists:
```python
gully_names = ["Heil-Ranch-gully1", "Heil-Ranch-gully2", "Geer-Canyon", "Central-Gulch"]
gully_burnseverity = ["severe", "moderate", "severe", "moderate"]
gully_nomral_I = [16, 16, 22, 61]           # normal infiltration rates in mm/hr
gully_postfire_I = [2.25, 11.77, 3.5, 4.0]  # 3 months post fire infiltration rates in mm/hr
```
### **New Tool**: grouped bars
To compare **two** values for each gully side by side, draw two sets of bars and shift them left and right of each gully's spot. The trick is to use **numbers** for the x positions, then put the names back on as tick labels:
```python
x = np.arange(4)       # 0, 1, 2, 3: one spot per gully
width = 0.4            # each bar is 0.4 wide, so two fit in one spot

fig, ax = plt.subplots()
ax.bar(x - width / 2, first_values, width=width, label="first")   # shifted left
ax.bar(x + width / 2, second_values, width=width, label="second") # shifted right
ax.set_xticks(x, gully_names)                                     # put a name under each spot
```
`ax.set_xticks(positions, labels)` says **where** the ticks goe and **what** to write at each one.

<div class="alert alert-info">

### Exercise 2 : infiltration before and after the fire

1. Copy the four lists above into your notebook.
2. Make a grouped bar chart: for each gully, one bar for the normal infiltration rate and one for the post-fire rate. Use two colors, a legened, and the gully names on the x-axis.
3. Pass the checklist. (Long names? tryp (`ax.tick_params(axis="x", labelrotation=20)`.)
    
</div>

In [3]:
# your answer here

<div class="alert alert-warning">

### Question 2
Which gully's infiltration dropped the **least**? look up its burn severity in `gully_burnseverity`. Now look at the **other** gully with the same classification. Is burn severity alone enough to predict ho wmuch infiltration drops?
    
</div>

> your answer here

### **New Tool**: a log-log plot of a threshold
At the end of Lab 02, you used the **Caine (1980)** threshold for debris flows:

$$I = 14.82\,D^{-0.39}$$

where $D$ is the storm duration (hours) and $I$ is the rainfall intensity (mm/hr) above which debris flows become likely. You tested five storms one at a time:

|**storm**|**intensity (mm/hr)**|**duration**|
|:--|--:|--:|
|storm 1|30|1|
|storm 2|18|0.2|
|storm 3|45|3|
|2013, period 1| 67| 12.5|
|2013, period 2|39|8|

*(The two 2013 periods are from the September 2013 Colorado Front Range storm, Coe et al., 2016)*

A power law like this one is a **straight line** on a log-log plot, and that's how Caine actually drew it. You draw on log-log axes with `ax.loglog(x, y)` in place of `ax.plot(x, y)` and anything else you add to the same axes (like a scatter) uses the same log axes:
```python
ax.loglog(durations, threshold, color="blac", label="Caine (1980) threshold")
ax.scatter(storm_D, storm_I)
```
To write a short label next to a point, use `ax.text(x, y, "words")`. It puts the **start** of the text at `(x, y)`

<div class="alert alert-info">

### Exercise 3: which storms cross the line?
1. Make `durations = np.linspace(0.1, 20, 200)` and compute the threshold `threshold` for all of them at once.
2. Put the five storms into **arrays**: `storm_I`, `storm_D`, and a list of names `storm_names`.
3. Make a mask `above` that is `True` for storms whos intensity is **at or above** the threshold for their own duration. (Compute the threshold at `storm_D`.)
4. On log-log axes, draw the threshold as a line. Then draw the storms **above** the line with one marker and color, and the storms **below** it with another, each with its own `label=`.
5. Use a `for` loop over `zip()` to write each storm's name next to its point with `ax.text()`
6. Pass the checklist.
    
</div>

<details>
<summary style="color:green;"><strong>Hint.</strong></summary>

#4. `storm_D[above]`, `storm_D[~above]`

In [4]:
# your answer here

<div class="alert alert-warning">

### Question 3
Storm 2 (18 mm/hr) is the only storm below the line. 2013 period 2 was only about twice as intense (39 mm/hr), yet it sits far above the line. Use the **shape** of the threshold line to explain why a **short** storm needs a **higher** intensity to trigger a debris flow than a long one.
    
</div>

> your answer here

<div class="alert alert-danger">

### CHECKPOINT: commit and push

Save your notebook, then:
```
git add lab09_LastName.ipynb
git commit -m "Finish Stops 1-2"
git push
```
    
</div>

---
## Stop 3: Lab 03, the factor of safety
In Lab 03, you wrote `factor_of_safety()` with the `math` module. `math.sin()` only works on **one number** at a time, so you had to call your function over and over to see how FS changed with slope. Swap `math.` for `np.`, and the same function works on a **whole array** at once:

In [5]:
def factor_of_safety(cohesion, soil_depth, soil_density, slope_deg, friction_deg, rw,
                     gravity=9.81, water_density=1000):
    """Infinite-slope factor of safety (Lab 03). Works on numbers or NumPy arrays."""
    theta = np.radians(slope_deg)
    phi = np.radians(friction_deg)

    A = cohesion / (soil_depth * soil_density * gravity * np.sin(theta))
    B = np.cos(theta) * np.tan(phi) * (1 - rw * water_density / soil_density) / np.sin(theta)

    return A + B

### **New Tool**: shading a region

`ax.fill_between(x, y1, y2)` shades between two curves. With two **numbers** for `y1` and `y2`, it shades a horizontal band across the whole plot, a nice way to makre a **danger zone**:
```python
ax.fill_between(sloeps, 0, 1, color="tab:red", alpha=0.15, label="Unstable (FS < 1)")
```


<div class="alert alert-info">

### Exercise 4: when does a wet slope fail?

Use the "medium" exabple from Lab 03:
- cohesion = 8500 Pa
- soil depth = 1.0 m
- soil density = 2000 kg/m$^3$
- friction angle = 32 deg

1. Make `slopes = np.linspace(5, 45, 81)` (in degrees).
2. Compute `fs_dry` (relative wetness `rw=0.1`) and `fs_wet` (`rw=1.0`) for **all** the sloeps in one call each.
3. Plot both as lines. Shade the unstable zone (FS < 1) with `fill_between`, and draw a line at FS = 1.
4. FS gets very large on gentle slopes. Zoom the y-axis to 0-4 so the interesting part is visible.
5. Use a mask to find the **first** slope where `fs_wet` drops below 1. Does `fs_dry` ever drop below 1 in this range?
6. Pass the checklist.

</div>

In [6]:
# your answer here

<div class="alert alert-warning">

### Question 4
The same hillslope is stable at 42$^\circ$ when dry but unstable when saturated. In Lab 03, you said relative wetness is the most climate-sensitive input. Using your figure, explain why a single big storm can turn a stable slope into a landslide.
    
</div>

> your answer here

<div style="background-color:#e8daef; border:1px solid #d2b4de; color:#6c3483; padding:10px; border-radius:4px; margin-bottom:20px;">

### Exercise G1 (graduate students; bonus for undergraduates, can replace 1 exercise): a stability map

How does FS depend on slope **and** cohesion together? Make a 2D map.
1. Make `cohesions = np.linspace(0, 20000, 101)` (Pa). Use broadcasting to compute `fs_map`, the saturated FS for every combination: pass `cohesions[:, np.newaxis]` and `slopes`.
2. Draw it with a color for each value:
3. 

---
## Stop 4: Lab 04, Nashville's precipitation normals

In the Lab 04 worksheet, you used Nashville's 1991-2020 monthly precipitation normals:
```python
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
mean_precip_in = [4.02, 4.47, 4.52, 4.72, 5.02, 4.36, 4.16, 3.79, 3.80, 3.36, 3.86, 4.43]
days_with_precip = [10.8, 10.9, 11.6, 11.2, 11.6, 10.7, 10.3, 9.4, 7.8, 8.4, 9.0, 11.4]
```
*(NOAA NCEI U.S. Climate Normals 1991–2020, station USW00013897, Nashville International Airport.)*

`mean_precip_in` is how **much** it rains in a typical month (inches). `days_with_precip` is how **often** (days with at least 0.01 in). They have different units, so the checklist says: different axes.

### **New Tool**: a second y-axis

`ax.twinx()` amkes a **second** axis on top of the first one. It shared the x-axis, but has its own y-axis on the *8right**:
```python
ax2 = ax.twinx()
ax2.plot(months, days_with_precip, color="black", marker="o")
ax2.set_ylabel("Days with precipitation", color="black")
```
A second y-axis is the one exception to checklist item 4, and its easy to misuse. Two general rules:
- **color each y-axis label to match it's data**, so the reader knows which axis goes with which.
- **start both axes at zero**. The two scales are independent, so where the line crosses the bars emans **nothing**, and you can make them cross wherever you like by changing the limits.

`ax.tick_params(axis="y", labelcolor="tab:cyan")` colors the tick numbers too.

<div class="alert alert-info">

### Exercsie 5 : How much, and how often?

1. Copy the three lists above.
2. Draw `mean_precip_in` as bars on `ax`, and `days_with_precip` as a line with markers on `ax2 = ax.twinx()`.
3. Color each y-axis label (and its tick numbers) to match its data. Start both y-axes at 0.
4. There are two kinds of data but only on legend: skip the legend and let the colored axis labels to the job.
5. Title, units: pass the checklist.
    
</div>

In [ ]:
# your answer here

<div class="alert alert-warning">

### Question 5
Is the month with the most rain also the month with the most rainy days? Is the driest month the one with the fewest rainy days? What does that say about the difference between **how much** and **how often**?
    
</div>

> your answer here

### Review: nested loops and `break`, drawn on a figure

In the Lab 04 worksheet, you found the **first** month, in calendar order, whose normal precipitation reaches each of several tresholds. The recipe was a **nested loop**: the outer loop over thresholds, the inner loop over months, and `break` as soon as you fin done. Now draw the answer.

Two reminders:
- `ax.text()` and `ax.axhline()` work inside a loop: each pass adds one more line or label to the same axes.
- to know whether the inner loop found **anything**, use a **flag variable** (Lecture 13): set `found=False` before teh inner loop, set it to `True` right before `break`, and check it after the loop.

<div class="alert alert-info">

### Exercise 6: the first month to reach each threshold
1. Make `levels = [4.0, 4.5, 5.0, 5.5]` (in), and call your loop variable `level`
2. Draw the monthly normals (`mean_precip_in`) as bars, with the y-axis from 0 to 6.
3. Write a **nested loop**: for each threshold, go through the months **in calendar order** with `zip()`. At the **first** month that reaches the threshold:
    - print a sentence naming the month,
    - draw a dashed horizontal line at the threshold
    - label the line with `ax.text()`, saying which month reached it first. put the labels above a **low** bar, like `"Oct"`, so they don't sit on top of the bars
    - then `break`
4. Use a flag variable so that a threshold **no** month reaches prints a sentence saying so.
5. Pass the checklist.
    
</div>

In [ ]:
# your answer here

<div class="alert alert-warning">

### Question 6
Predict what would happen to the **figure** and the **printed output** if you deleted the **break**. Then try it and check.
    
</div>

> your answer here

<div class="alert alert-danger">

### CHECKPOINT: commit and push

Save your notebook, then:
```
git add lab09_LastName.ipynb
git commit -m "Finish Stops 3-4"
git push
```
    
</div>

### Halfway there
If you've reached this point by **Sunday, November 1**, you're on pace. Stops 5-8 are next week's work.

---
## Stop 5 : Lab 06, Nisqually and Northridge

Load the aftershock catalogs the same way as Lab 06.
```python
nis = np.loadtxt("nisqually_2001.csv", delimiter=",")
nor = np.loadtxt("northridge_1994.csv", delimiter=",")

nis_hours, nis_lat, nis_lon, nis_depth, nis_mag, nis_eq = nis[:, 0], nis[:, 1], nis[:, 2], nis[:, 3], nis[:, 4], nis[:, 5]
nor_hours, nor_lat, nor_lon, nor_depth, nor_mag, nor_eq = nor[:, 0], nor[:, 1], nor[:, 2], nor[:, 3], nor[:, 4], nor[:, 5]

# aftershocks: earthquakes, after the mainshock, in the first 7 days (Lab 06, Exercise 4)
nis_after = (nis_eq == 1) & (nis_hours > 0) & (nis_hours <= 168)
nor_after = (nor_eq == 1) & (nor_hours > 0) & (nor_hours <= 168)
```
### **New Tool**, (part 1 of log axes): a log axis for time
Aftershocks come fast at first, then slow down. On an ordinary time axis, the first hour, when the most is happening, is squeezed into a sliver at the left edge. A **log** time axis gives the first minute, the first hour, and the first day equal room:
```python
ax.set_xscale("log")
```
Times have to be **greater than 0** on a log axis, which our aftershock mask already guarantees.

<div class="alert alert-info">

### Exercise 7: aftershocks through time

1. Copy the loading code above, but be sure you find the correct file locations (should be in the same folder that Lab 06 was in).
2. Make a figure with **1 row and 2 columns** that **share the y-axis**. Left: Nisqually. Right: Northridge.
3. In each, scatter the aftershocks' **magnitude** against **hours after the mainshock**, with a log x-axis. With thousands of Northridge points, use small, see-through dots.
4. Give each panel a title, label every axis, and finish with `fig.tight_layout()`
    
</div>

In [7]:
# your code here

<div class="alert alert-warning">

### Question 7
On the Northridge panel, read the x-axis carefully: each tick is 10 times the one before. Roughly how many **minutes** after the mainshock did the first aftershocks happen? Why would this be nearly invisible on an ordinary (linear) time axis that runs to 168 hours?
    
</div>

> your answer here

### **New Tool**, (part 2 of log axes): Gutenberg-Richter on a log axis

Seismologists found that each step **down** in magnitude brings many **more** earthquakes, by about the same factor each time (Gutenberg & Richter, 1944). A histogram of magnitudes with a log y-axis shows this as a nearly straight line.

<div class="alert alert-info">

### Exercise 8: how common are big aftershocks?

1. Make `nor_after_mag`, Northridge's aftershock magnitudes, with the `nan`s removed.
2. Draw a histogram of `nor_after_mag` with `bins=np.arange(0, 6.25, 0.25)` (bins a quarter of a magnitude wide) and put the y-axis on a log scale.
3. Count the aftershocks of M2 or more, M3 or more, M4 or more, and M5 or more. Print them.
    
</div>

In [8]:
# your answer here

<div class="alert alert-warning">

### Question 8
On the log axis, the right side of the histogram is close to a straight line, but the left side **falls off** below about magnitude 1.5. Small earthquakes are actually the **most** common. So why might the catalog have fewer of them?
    
</div>

> your answer here

### Review: 2D arrays and `axis=`

`nor` is a **2D array**: one row per event, one column per measurement. Using a mask on the **rows**, `nor[nor_after]`, keeps it 2D: only the aftershock rows, all six columns.

Remember from Lecture 16: the `axis` number matches a position in `.shape`. For a `(rows, columns)` array, `axis=0` uses the **rows**, so you get **one answer per column**

<div class="alert alert-info">

### Exercise 9: every column at once

1. Make `after = nor[nor_after]`. Print its `.shape`. How many rows and columns?
2. Use `np.nanmin(after, axis=0)` and `np.nanmax(after, axis=0)` to get the smallest and largest value in **every column** in two lines.
3. Use a `for` loop with `zip()` over a list of column names, the mins, and the maxes to print one neat line per column.
4. Find the **depest** aftershock: use `np.nanargmax()` on the depth column, then prin that event's whole **row** with `after[i]`. How long after the mainshock was it, and how big?
    
</div>

In [ ]:
# your answer here

### **New Tool**, (part 3 of log axes): a map colored by a third variable
A scatter plot can show **three** things at ocne: `x`, `y`, and a **color**. Pass the values for the color with `c=`, choose a colormap with `cmap=`, and add a **colorbar** so the reader can read the colors:
```python
sc = ax.scatter(x, y, c=values, cmap="viridis", s=5)
fig.colorbar(sc, ax=ax, label="what the color means (units)")
```
`ax.scatter()` hands back the dots as an object (here `sc`); the colorbar needs it to know which colors mean what. That's the same ideas as `fig, ax`: keep a name for the thing you'll need later.

For a **map**, put longitudde on x and latitude on y. One degree of longitude is shorter than one degree of latitude away from the equator, so tell matplotlib to stretch the axes to match:
```python
ax.set_aspect(1 / np.cos(np.radians(34.2))) # 34.2 = the latitude of the map
```

<div class="alert alert-info">

### Exercise 10: where were Northridge's aftershocks?

1. Make a map of Northridge's aftershocks: longitude on x, latitude on y, **each dot colored by depth.** Use small dots, a colormap of your choice, and a colorbar with units.
2. Mark the **mainshock epicenter** (34.213N, 118.537W, from the file header) with a big star, and give it a label and a legend.
3. Set the aspect ratio with the line above, and pass the checklist (Longitude is negative for west).
    
</div>

In [9]:
# your answer here

<div class="alert alert-warning">

### Question 9
Is the mainshock in the **middle** of the aftershock cloud, or off to one side? where are the deepest aftershocks? What might tell you about which part of the fault slipped?
    
</div>

> your answer here

<div class="alert alert-danger">

### CHECKPOINT: commit and push

Save your notebook, then:
```
git add lab09_LastName.ipynb
git commit -m "Finish Stop 5"
git push
```
    
</div>

---
## Stop 6: Lab 08, the ocean buoys

Your Assignment 2 repo has **cleaned** versions of the two buoy files from lab 08, with the same cleaning you did in Lab 08: `-999` replaced by blanks, pH kept only where its quality flag was 2 (good), and a `DATETIME_UTC` column. The first three lines of each file are notes that start with `#`; `comment="#"` tells pandas to skip them:
```python
kan = pd.read_csv("kaneohe_clean.csv", comment="#", parse_dates=["DATETIME_UTC"], index_col="DATETIME_UTC")
ore = pd.read_csv("nh10_clean.csv", comment="#", parse_dates=["DATETIME_UTC"], index_col="DATETIME_UTC")
```
### **New Tool**: error bars
A monthly **mean** hides how much the values **spread** around it. Error bars show both: a point at the mean and a bar reaching one standard deviation above and below.
```python
ax.errorbar(x, means, yerr=stds, fmt="o-", capsize=4, label='...')
```
- `yerr=` is how far each bar reaches above **and** below its point
- `fmt="o-"` means "circles connected by a line"
- `capsize=` puts little caps on the ends of the bars

**Careful:** here, the bars show how much pH **varies** within a month, **not** how uncertain the mean is. Always say which on your error bars mean, in a caption or legend.

<div class="alert alert-info">

### Exercise 11: Monthly pH at both sites

1. Load both files. Print `.describe()` of each site's `pH` column for a first look.
2. For each site, use `groupby` on the month to get the monthly `mean`, `std`, and `count` of `pH`. You'll need three `groupby` lines per site.
3. Some months have very few good pH values (or none). Make a mask that keeps only months with **at least 100** good values, and use it on the means, the stds, and the index.
4. Draw both sites on **one** axes with `ax.errorbar()` using different markers and colors, and a legend that says what the bars mean.
5. Put month **names** on teh x-axis with `ax.set_xticks(range(1, 13), months)` (the `months` list from Stop 4), and pass the checklist.
6. Save Kāneʻohe's monthly means to a file, `kaneohe_monthly_pH.csv`, with `.to_csv()` (Lecture 19). A Series can be saved thes ame way as a DataFrame.
    
</div>

In [10]:
# your answer here

<div class="alert alert-warning">

### Question 10
Which site's error bars are longer? connect that to what you found about the **daily** pH cycle in Lab 08. Then: if you only had the monthly **menas8* (no error bars), what would you have missed?
    
</div>

> your answer here

<div class="alert alert-info">

### Exercise 12: the reef's day, seen two ways
1. Make `kan_local` with teh code above.
2. Add a **new column**, `kan_local["hour"]`, holding the local hour of each measurement (`kan_local.index.hour`).
3. Make a figure with **1 row and 2 columns** that **share the y-axis**:
    - **left**: zoom into one week with a `.loc` date slice, `kan_local.loc["2016-01-20":"2016-01-16"]`, and plot its pH against time with markers (`"o-"`). Tilt the date labels.
    - **right**: the **average day**: `groupby` the `"hour"` column for the mean pH at each hour, `.dropna()` it, and plot it with markers.
4. Use `.idxmax()` and `.idxmin()` on the hourly means to find the hours of the highest and lowest pH. Mark the peak on teh right panel with `ax.annotate()`.
5. label everything (the x-axis on teh right is "Hour of day, Hawai'i time"), and finish with `fig.tight_layout()`.
    
</div>

In [11]:
# your answer here

<div class="alert alert-warning">

### Question 11
The average day (right) swings much less than the invididual days in teh week on the left. Why does averaging make the cycle look smaller? If you were writing about how much the reef's chemistry changes in a day, which panel would you use, and why?
    
</div>

> your answer here

<div style="background-color:#e8daef; border:1px solid #d2b4de; color:#6c3483; padding:10px; border-radius:4px; margin-bottom:20px;">

### Exercise G2 (graduate students; bonus for undergraduates, can replace 1 exercise): pH and CO$_2$ colored by temperature

In Lab 08, positive `dpCO2` mean the seawater held more CO$_2$ than the air above it. More dissovled CO$_2$ should mean **lower** pH.

1. On one axes, scatter **pH** (y) against **dpCO2** (x) for **both** sites, colored by `SST` (`c=`), with a colorbar labeled with units. use a different **marker** for each site. A legend doesn't work well for colored scatters (try it and see), so label each site's cloud with `ax.text()` instead.
2. Both scatters musst use the **same** color scale, or the same color will mean different temperatures. Pass `vmin=` and `vmax=` to both. (Look at the range of `SST` at both sites to choose them.)
3. Use `.corr()` to find the correlation between pH and dpCO2 at each site.
4. **Quetion**: Is the relationship negative at both sites? Which site is it stronger at? Why might the colors separate the two sites so cleanly?


In [12]:
# your answer here

---
## Stop 7: a schematic, from an idea (Lecture 22)

Every stop so far started with data. This one starts wtih an **idea**. Pick **one** of these ideas from earlier labs:
|**idea**|**from**| **the story to draw**|
|:--|:--|:--|
|**(a) A reef's pH, day and night**| Labs 06 and 08| On a coral reef, pH rises during the day (photosynthesis takes up CO$_2$) and falls at night (respiration adds it back). Underneath that daily cycle, ocean acidificatino slowly lowers the average.|
|**(b) A hillslope through a storm**| Lab 03| Before a storm the slope is dry and stable. As rain soacks in, relative wetness rises and the factor of safety drops, maybe below 1. After the storm, the slope sloly drains and FS recovers.|
|**(c) a burned watershed over years**| Lab 02| infiltration is steady, then **drops suddenly** when a fire burns, then **slowly recovers** over the following years as plants grow back.|

Follow Lecture 22's recipe: **sketch**, **pieces**, **arrays**, **style**.

<div class="alert alert-info">

### Exercise 13: From sketch to schematic

1. **Sketch it on paper first.** What's on each axis? what shape does each curve take? Where do the labels go? Take a photo of your sketch and it it to a Markdown cell (drag the image into the cell), or describe it in words.
2. In a Markdown cell, make a  **table like the one in Lecture 22, Part 2:** each shape in your sketch, what it means, and what you'll type (Useful piece: a straightline:, `start + slope * x`; a jump, `y[x >= where] += amount`; decay or recover, `np.exp(-x / length)`; a repeating cycle, `np.sin(2 * np.pi * x)` from Lab 06).
3. Build the arrays **one piece at a time**, plotting after each piece. Put the "knobs" (sizes, locations, decay lengths) in variables.
4. Style it as a schematic: no tick numbers, top and right spines hidden, labels that point at features (`ax.annotate()` or `ax.texct()`) instead of a legend, and "**schematic**" in teh title.
5. Save it as `assignment02_schematic_LastName.png` at 300 dpi and make sure you submit it to your assignment repo.
    
</div>

In [13]:
# your answer here

<div class="alert alert-danger">

### CHECKPOINT: commit and push

Save your notebook, then:
```
git add lab09_LastName.ipynb
git commit -m "Finish Stop 6-7"
git push
```
    
</div>

---
## Stop 8: one figure, one story

Scientists rarely show every figure they make. In a paper or a talk, they combine a few panels into **one** figure that makes **one** point, with a caption that tells the reader what to notice.

Some stories you could tell with what you've made:
- **Water is the trigger**. Fire cuts infiltration (Stop 2), intense or long storms cross the debris-flow threshold (Stop 2), and wet soil fails at gentler slopes (Stop 3).
- **Two earthquakes, two personalitites**. Nisqually and Northridge through time (Stop 5), and where Northridge's aftershocks were (Stop 5).
- **Two oceans**. pH at each buoy by month (Stop 6), and how pH relates to CO$_2$ (G2), or your reef schematic (Stop 7).
- 
Or make up your own

<div class="alert alert-info">

### Exercise 14: a multi-panel story figure

1. In a Markdown cell, wirte your story in **one sentence**. That's your figure's main point.
2. Make **one** figure with at least **2 panels** (`plt.subplots()`), using plots from at least **two** different stops. You can change them: re-color, re-label, zoom, simplify, whatever helps the story.
3. Give the figure an overall title with `fig.suptitle()`, label the panels **(a)**, **(b)**, **(c)** in their titles, share axes where panels are meant to be compared, and use `fig.tight_layout()`.
4. In a Markdown cell, go through the **7-item checklist** and say in one line each how your figure passes it.
5. Write a **caption** of 3-5 sentences, the way it would appear under the figure in a peper: what each panel shows, where the data came from, and the one thing the reader should take away.
6. Save it as `assignment2_story_LastName.png` at **300 dpi** with `bbox_inches="tight"`, and make sure you submit it to your assignment repo.
    
</div>

In [ ]:
# your answer here

# End for Everyone

## Submitting

Save your notebook (ctrl + S), then in the terminal from your Assignment 2 repo folder:
```
git status
git add .
git commit -m "[insert a message here]"
git push
```
Check your repo's page on GitHub to make sure your commit arrived, including both PNGs. You can push as many tiems as you like before the deadline; your most recent push is what counts.

## Quick reference: the new tools in this assignment

|to...|write|stop|
|:--|:--|:--|
|grouped bars|`ax.bar(x - w/2, a, width=w)`, `ax.bar(x + w/2, b, width=w)`, `ax.set_xticks(x, names)`|2|
|log-log|`ax.loglog(x, y)`|2|
|text next to a point|`ax.text(x, y, "words")`|2|
|shade a band|`ax.fill_between(x, y1, y2, alpha=...)`|3|
|second y-axis|`ax2 = ax.twinx()`|4|
|color tick labels|`ax.tick_params(axis="y", labelcolor=...)`|4|
|log time axis|`ax.set_xscale("log")`|5|
|scatter colored by a value|`sc = ax.scatter(x, y, c=values, cmap=...)`, `fig.colorbar(sc, ax=ax, label=...)`|5, 6|
|map aspect|`ax.set_aspect(1 / np.cos(np.radians(lat)))`|5|
|error bars|`ax.errorbar(x, y, yerr=..., fmt="o-", capsize=4)`|6|
|color grid (grad)|`ax.pcolormesh(x, y, values)`, `ax.contour(..., levels=[1])`|3|
|a table from two 1D arrays|`f(a[:, np.newaxis], b)`|3|
|hide box lines|`ax.spines["top"].set_visible(False)`|7|
|no tick numbers|`ax.set_xticks([])`, `ax.set_yticks([])`|7|
|panel labels and overall title|`ax.set_title("(a) ...")`, `fig.suptitle("...")`|6, 8|
|lines and labels from a loop|`ax.axhline()`, `ax.text()` inside a `for` loop|4|

## References
- Caine, N. (1980). The rainfall intensity-duration control of shallow landslides and debris flows. *Geografiska Annaler: Series A, Physical Geography*, 62(1-2), 23-27. https://doi.org/10.1080/04353676.1980.11879996
- Coe, J. A., Kean, J. W., Godt, J. W., Baum, R. L., Jones, E. S., Gochis, D. J., & Anderson, G. S. (2016). New insights into debris-flow hazards from an extraordinary event in the Colorado Front Range. *GSA Today*, 24(10), 4-10. https://doi.org/10.1130/GSATG214A.1
- Gutenberg, B., & Richter, C. F. (1944). Frequency of earthquakes in California. *Bulletin of the Seismological Society of America*, 34(4), 185–188.
- NOAA National Centers for Environmental Information, U.S. Climate Normals 1991–2020, station USW00013897 (Nashville Intl AP, TN).
- NOAA Pacific Marine Environmental Laboratory, ocean acidification moorings Kāneʻohe (158°W, 21°N) and NH10 (124°W, 44°N), as used in Lab 08.
- U.S. Geological Survey, ANSS Comprehensive Earthquake Catalog (ComCat), accessed October 11, 2026. https://earthquake.usgs.gov/data/comcat/
- VanderPlas, J. *Python Data Science Handbook*, Chapter 4. https://jakevdp.github.io/PythonDataScienceHandbook/04.00-introduction-to-matplotlib.html